# E-Commerce Sales Analysis — Summary Charts

This notebook connects to the local **PostgreSQL** database `olist_analytics`
and produces three summary visuals with Plotly:

1. Monthly revenue and order count
2. Top-10 categories by revenue
3. Delivery timeliness vs. average review score

All metrics use **delivered** orders and revenue = `SUM(order_items.price)`.
The heavy lifting is done in SQL (`sql/01`–`sql/07`); here we only visualize
the aggregated results.


## 1. Setup and database connection

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text

# --- Connection settings -------------------------------------------------
# Postgres.app default: local socket, your macOS username, no password.
# Override via environment variables if needed.
DB_USER = os.getenv("PG_USER", os.getenv("USER", "postgres"))
DB_PASSWORD = os.getenv("PG_PASSWORD", "")
DB_HOST = os.getenv("PG_HOST", "localhost")
DB_PORT = os.getenv("PG_PORT", "5432")
DB_NAME = os.getenv("PG_DATABASE", "olist_analytics")

auth = f"{DB_USER}:{DB_PASSWORD}" if DB_PASSWORD else DB_USER
engine = create_engine(f"postgresql+psycopg2://{auth}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Verify the connection
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).scalar()
print("Connected to:", version)

# Folder for exported PNG charts
IMAGES_DIR = Path("..") / "images"
IMAGES_DIR.mkdir(exist_ok=True)


## 2. Chart 1 — Monthly revenue and orders

Revenue (bars) and order count (line) per month for delivered orders.
Note: the first months of 2016 and the last months of 2018 are sparse/incomplete
and should be read with caution.


In [ ]:
monthly_sql = """
WITH delivered_orders AS (
    SELECT o.order_id,
           DATE_TRUNC('month', o.order_purchase_timestamp)::date AS order_month
    FROM raw.orders o
    WHERE o.order_status = 'delivered'
),
order_revenue AS (
    SELECT order_id, SUM(price) AS revenue
    FROM raw.order_items
    GROUP BY order_id
)
SELECT d.order_month,
       COUNT(DISTINCT d.order_id) AS orders_count,
       ROUND(SUM(r.revenue), 2)   AS revenue
FROM delivered_orders d
JOIN order_revenue r ON d.order_id = r.order_id
GROUP BY d.order_month
ORDER BY d.order_month;
"""

monthly = pd.read_sql(monthly_sql, engine)
monthly["order_month"] = pd.to_datetime(monthly["order_month"])
monthly.head()


In [ ]:
fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(
    go.Bar(x=monthly["order_month"], y=monthly["revenue"],
           name="Revenue (R$)", marker_color="#4C78A8"),
    secondary_y=False,
)
fig1.add_trace(
    go.Scatter(x=monthly["order_month"], y=monthly["orders_count"],
               name="Orders", mode="lines+markers", line=dict(color="#F58518")),
    secondary_y=True,
)

fig1.update_layout(
    title="Monthly Revenue and Orders (Delivered)",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    width=900, height=500,
)
fig1.update_xaxes(title_text="Month")
fig1.update_yaxes(title_text="Revenue (R$)", secondary_y=False)
fig1.update_yaxes(title_text="Orders", secondary_y=True)

fig1.write_image(IMAGES_DIR / "monthly_revenue.png", scale=2)
fig1.show()


## 3. Chart 2 — Top-10 categories by revenue

Category names are translated to English via `product_category_name_translation`.


In [ ]:
category_sql = """
WITH delivered_items AS (
    SELECT oi.order_id,
           oi.price,
           COALESCE(t.product_category_name_english,
                    p.product_category_name,
                    'unknown') AS category
    FROM raw.order_items oi
    JOIN raw.orders o
        ON oi.order_id = o.order_id
        AND o.order_status = 'delivered'
    LEFT JOIN raw.products p ON oi.product_id = p.product_id
    LEFT JOIN raw.category_translation t
        ON p.product_category_name = t.product_category_name
)
SELECT category,
       ROUND(SUM(price), 2) AS revenue
FROM delivered_items
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
"""

top_categories = pd.read_sql(category_sql, engine)
top_categories


In [ ]:
fig2 = px.bar(
    top_categories.sort_values("revenue"),
    x="revenue", y="category", orientation="h",
    title="Top-10 Categories by Revenue (Delivered)",
    labels={"revenue": "Revenue (R$)", "category": "Category"},
    color="revenue", color_continuous_scale="Blues",
)
fig2.update_layout(template="plotly_white", width=900, height=500,
                   coloraxis_showscale=False)

fig2.write_image(IMAGES_DIR / "category_revenue.png", scale=2)
fig2.show()


## 4. Chart 3 — Delivery timeliness vs. review score

Average review score by delivery group (on-time/early vs. late).

**Note:** this shows a *correlation*, not proven causation. Other factors
(category, region, price) may also influence review scores.


In [ ]:
delivery_sql = """
WITH order_delivery AS (
    SELECT order_id,
           EXTRACT(EPOCH FROM (order_delivered_customer_date
                   - order_estimated_delivery_date)) / 86400.0 AS delay_days
    FROM raw.orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
),
order_review AS (
    SELECT order_id, AVG(review_score::numeric) AS review_score
    FROM raw.order_reviews
    GROUP BY order_id
),
joined AS (
    SELECT CASE WHEN d.delay_days > 0 THEN 'Late'
                ELSE 'On time / Early' END AS delivery_group,
           r.review_score
    FROM order_delivery d
    JOIN order_review r ON d.order_id = r.order_id
)
SELECT delivery_group,
       COUNT(*)                    AS orders,
       ROUND(AVG(review_score), 3) AS avg_review_score
FROM joined
GROUP BY delivery_group
ORDER BY avg_review_score DESC;
"""

delivery = pd.read_sql(delivery_sql, engine)
delivery


In [ ]:
fig3 = px.bar(
    delivery, x="delivery_group", y="avg_review_score",
    title="Average Review Score by Delivery Timeliness",
    labels={"delivery_group": "Delivery", "avg_review_score": "Avg review score"},
    color="delivery_group",
    color_discrete_map={"On time / Early": "#54A24B", "Late": "#E45756"},
    text="avg_review_score",
)
fig3.update_traces(textposition="outside")
fig3.update_layout(template="plotly_white", width=700, height=500,
                   showlegend=False, yaxis_range=[0, 5])

fig3.write_image(IMAGES_DIR / "delivery_vs_reviews.png", scale=2)
fig3.show()


## 5. Takeaways

- Growth was driven by **acquisition, not retention** (repeat rate 3.00%).
- **Late deliveries** coincide with a sharp drop in review scores (4.29 → 2.57);
  correlation, not proven causation.
- Revenue is **moderately concentrated** in a handful of leading categories.

See the main `README.md` for the full findings and business recommendations.
